In [ ]:
# =========================================================================
# 1. IMPORTS AND CONFIGURATION
# =========================================================================

# -------------------------------------------------------------------------
# 1.1. Standard Library & Third-Party Imports
# -------------------------------------------------------------------------
from dataclasses import dataclass
import shutil
from pathlib import Path
from typing import Tuple

import h5py
import numpy as np
import rasterio
from tqdm import tqdm


In [ ]:
# -------------------------------------------------------------------------
# 1.2. Configuration Dataclass
# -------------------------------------------------------------------------
# Canonical layout (see README, Docs/data.md, scripts/check_data.py):
#   hdf5_processed/{Source,Target_Original,Target_Low,Target_Medium,Target_High}/
# Processing runs on local disk (/tmp): the Drive FUSE mount truncates
# large ~900 MB reads (TIFFFillTile / IReadBlock); the full tree is published
# back to Drive at the end.
DRIVE_ROOT = Path("/content/drive/MyDrive/GEE_DARES_Dataset")
LOCAL_ROOT = Path("/tmp/dares_work")
LOCAL_TIF_DIR = LOCAL_ROOT / "tifs"
LOCAL_H5_DIR = LOCAL_ROOT / "hdf5_processed"
DRIVE_H5_DIR = DRIVE_ROOT / "hdf5_processed"

# Domain -> (subfolder, filename prefix)
DOMAIN_LAYOUT = {
    "source": ("Source", "source"),
    "target": ("Target_Original", "target"),
}


@dataclass(frozen=True)
class DatasetConfig:
    # Roots (single source of truth; everything else derives from here)
    drive_root: Path = DRIVE_ROOT
    local_root: Path = LOCAL_ROOT

    # GeoTIFF filenames on Drive
    source_image: str = "source_brasil_s2_2021_10m.tif"
    source_gt: str = "source_brasil_gt_2021_10m.tif"
    target_image: str = "target_colombia_s2_2021_10m.tif"
    target_gt: str = "target_colombia_gt_2021_10m.tif"

    @property
    def input_dir(self) -> Path:
        return self.local_root / "tifs"

    @property
    def local_hdf5_dir(self) -> Path:
        return self.local_root / "hdf5_processed"

    @property
    def drive_hdf5_dir(self) -> Path:
        return self.drive_root / "hdf5_processed"

    # Patch extraction parameters
    patch_size: int = 224
    stride: int = 112

    # Quality and entropy filtering thresholds (max_nodata_ratio = max 255/invalid ratio)
    max_nodata_ratio: float = 0.10
    min_forest_ratio: float = 0.15
    max_forest_ratio: float = 0.85

    # Spatial block splitting ratios along Y-axis (Rows)
    train_ratio: float = 0.70
    val_ratio: float = 0.20
    test_ratio: float = 0.10


# =========================================================================
# 2. RASTER I/O AND PREPROCESSING UTILITIES
# =========================================================================


IGNORE_INDEX = 255


def load_raster_pair(
    image_path: Path, gt_path: Path
) -> Tuple[np.ndarray, np.ndarray]:
    """Reads image and ground truth GeoTIFF rasters into aligned NumPy arrays.

    Preserves water/wetland as 255 (ignore). Masks are {0, 1, 255}:
    1 = forest, 0 = non-forest, 255 = water/wetland/NoData (ignored).

    Synchronizes NoData masks and scales Sentinel-2 surface reflectance
    to the physical float interval [0.0, 1.0].

    Args:
        image_path: Path to the 4-band Sentinel-2 GeoTIFF.
        gt_path: Path to the 1-band Ground Truth GeoTIFF (values 0, 1, 255).

    Returns:
        Tuple of (image_array, mask_array):
            - image_array: np.ndarray of shape (4, H, W), float32.
            - mask_array: np.ndarray of shape (H, W), int16 with 255 on ignore.
    """
    with rasterio.open(image_path) as src_img:
        image = src_img.read().astype(np.float32)  # Shape: (4, H, W)
        img_nodata = src_img.nodata

    with rasterio.open(gt_path) as src_gt:
        mask = src_gt.read(1).astype(np.int16)  # Shape: (H, W), values 0, 1, 255
        gt_nodata = src_gt.nodata

    # Image NoData -> NaN (same as before)
    if img_nodata is not None:
        image[:, image[0] == img_nodata] = np.nan

    # GT: any real nodata -> 255 (ignore), NEVER -> 0
    if gt_nodata is not None:
        mask[mask == gt_nodata] = IGNORE_INDEX

    # Where the image has no valid data, the label is also ignored
    img_invalid = np.isnan(image).any(axis=0)
    mask[img_invalid] = IGNORE_INDEX

    # Normalize Sentinel-2 surface reflectance from [0, 10000] to [0.0, 1.0]
    # (only over valid image pixels; ignored pixels do not affect scaling)
    valid_pixels = image[:, ~img_invalid]
    if valid_pixels.size > 0 and np.nanmax(valid_pixels) > 10.0:
        image = image / 10000.0

    # Clip residual reflectance artifacts to strict [0.0, 1.0] interval
    image = np.clip(image, 0.0, 1.0)
    # Image NaNs -> 0 (the label there is already 255)
    image = np.nan_to_num(image, nan=0.0)

    return image, mask


# =========================================================================
# 3. PATCH EXTRACTION AND FILTERING PIPELINE
# =========================================================================


def extract_filtered_patches(
    image_block: np.ndarray, mask_block: np.ndarray, config: DatasetConfig
) -> Tuple[np.ndarray, np.ndarray]:
    """Extracts sliding window patches and applies water/NoData and entropy filters.

    Args:
        image_block: Input sub-array of shape (4, H_block, W).
        mask_block: Target sub-array of shape (H_block, W).
        config: Configuration instance with extraction parameters.

    Returns:
        Tuple of (images, masks):
            - images: np.ndarray of shape (N, 4, patch_size, patch_size), float32.
            - masks: np.ndarray of shape (N, patch_size, patch_size), uint8.
    """
    _, height, width = image_block.shape
    p_size = config.patch_size
    stride = config.stride

    extracted_images = []
    extracted_masks = []
    total_pixels = p_size * p_size

    row_coords = list(range(0, height - p_size + 1, stride))
    col_coords = list(range(0, width - p_size + 1, stride))

    for r in row_coords:
        for c in col_coords:
            img_patch = image_block[:, r : r + p_size, c : c + p_size]
            mask_patch = mask_block[r : r + p_size, c : c + p_size]

            # 1. Filter: ratio of INVALID (255 = water/wetland/NoData) < 10%
            invalid_count = (mask_patch == 255).sum()
            if (invalid_count / total_pixels) > config.max_nodata_ratio:
                continue

            valid = (mask_patch != 255)
            valid_count = valid.sum()
            if valid_count == 0:
                continue

            # 2. Filter: forest ratio in [15%, 85%] over VALID pixels only
            forest_count = (mask_patch[valid] == 1).sum()
            forest_ratio = forest_count / valid_count

            if not (
                config.min_forest_ratio <= forest_ratio <= config.max_forest_ratio
            ):
                continue

            # Sanitize image only; mask keeps 255 (uint8 preserves it)
            img_patch_clean = np.nan_to_num(img_patch, nan=0.0)

            extracted_images.append(img_patch_clean.astype(np.float32))
            extracted_masks.append(mask_patch.astype(np.uint8))

    if not extracted_images:
        return (
            np.empty((0, 4, p_size, p_size), dtype=np.float32),
            np.empty((0, p_size, p_size), dtype=np.uint8),
        )

    return np.stack(extracted_images, axis=0), np.stack(extracted_masks, axis=0)


# =========================================================================
# 4. HDF5 STORAGE AND PERSISTENCE
# =========================================================================


def save_to_hdf5(
    output_path: Path, images: np.ndarray, masks: np.ndarray
) -> None:
    """Saves image and mask arrays into a compressed HDF5 container.

    Args:
        output_path: Destination .h5 file path.
        images: Image tensor array (N, 4, 224, 224).
        masks: Mask tensor array (N, 224, 224).
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with h5py.File(output_path, "w") as f:
        f.create_dataset(
            "images",
            data=images,
            dtype="float32",
            chunks=(1, 4, images.shape[2], images.shape[3]),
            compression="lzf",
        )
        f.create_dataset(
            "masks",
            data=masks,
            dtype="uint8",
            chunks=(1, masks.shape[1], masks.shape[2]),
            compression="lzf",
        )


# =========================================================================
# 5. DOMAIN PROCESSING WORKFLOW
# =========================================================================


def process_domain(
    domain_name: str,
    image_filename: str,
    gt_filename: str,
    config: DatasetConfig,
) -> None:
    """Executes raster loading, Y-axis block partitioning, and HDF5 generation.

    Args:
        domain_name: Domain identifier ('source' or 'target').
        image_filename: GeoTIFF satellite image filename.
        gt_filename: GeoTIFF ground truth filename.
        config: DatasetConfig instance.
    """
    img_path = config.input_dir / image_filename
    gt_path = config.input_dir / gt_filename

    print(f"\n{'=' * 60}")
    print(f"Processing Domain: {domain_name.upper()} (Sentinel-2 10m)")
    print(f"{'=' * 60}")
    print(f"Loading: {img_path.name} and {gt_path.name}...")

    image, mask = load_raster_pair(img_path, gt_path)
    total_rows = image.shape[1]

    # Partition along Y-axis into Train (70%), Val (20%), Test (10%)
    train_end = int(total_rows * config.train_ratio)
    val_end = train_end + int(total_rows * config.val_ratio)

    try:
        subdir, prefix = DOMAIN_LAYOUT[domain_name]
    except KeyError:
        raise ValueError(f"domain_name must be one of {sorted(DOMAIN_LAYOUT)}, got {domain_name!r}")

    splits = {
        "train": (
            slice(0, train_end),
            config.local_hdf5_dir / subdir / f"{prefix}_train.h5",
        ),
        "val": (
            slice(train_end, val_end),
            config.local_hdf5_dir / subdir / f"{prefix}_val.h5",
        ),
        "test": (
            slice(val_end, total_rows),
            config.local_hdf5_dir / subdir / f"{prefix}_test.h5",
        ),
    }

    for split_name, (row_slice, out_file) in splits.items():
        sub_image = image[:, row_slice, :]
        sub_mask = mask[row_slice, :]

        patches_img, patches_mask = extract_filtered_patches(
            sub_image, sub_mask, config
        )
        save_to_hdf5(out_file, patches_img, patches_mask)

        print(
            f"  -> [{split_name.upper()}] Saved: {len(patches_img):4d} patches | "
            f"Tensor: {patches_img.shape} | File: {out_file.name}"
        )


def stage_inputs(config: DatasetConfig) -> None:
    """Copies the .tif files from Drive to local disk before processing."""
    config.input_dir.mkdir(parents=True, exist_ok=True)
    for fname in (
        config.source_image,
        config.source_gt,
        config.target_image,
        config.target_gt,
    ):
        src = config.drive_root / fname
        dst = config.input_dir / fname
        if not src.exists():
            raise FileNotFoundError(f"Not found on Drive: {src}")
        if dst.exists() and dst.stat().st_size == src.stat().st_size:
            print(f"  [stage] already local: {fname}")
            continue
        print(f"  [stage] copying {fname} ({src.stat().st_size/1024**3:.2f} GB)...")
        shutil.copyfile(src, dst)
    print(f"  [stage] done at {config.input_dir}")


def publish_outputs(config: DatasetConfig) -> None:
    """Publishes the local hdf5_processed/... tree back to Drive."""
    dest = config.drive_hdf5_dir
    for f in sorted(config.local_hdf5_dir.rglob("*.h5")):
        rel = f.relative_to(config.local_hdf5_dir)
        target = dest / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        print(f"  [publish] {rel} -> {target}")
        shutil.copyfile(f, target)
    print(f"  [publish] done at {dest}")


# =========================================================================
# 6. MAIN EXECUTION PIPELINE
# =========================================================================


def main_build_base() -> None:
    config = DatasetConfig()

    print(f"Staging inputs {config.drive_root} -> {config.input_dir}...")
    stage_inputs(config)

    # 1. Source Domain: Brazil (São Félix do Xingu - Sentinel-2)
    process_domain(
        domain_name="source",
        image_filename=config.source_image,
        gt_filename=config.source_gt,
        config=config,
    )

    # 2. Target Domain (Clean): Colombia (San José del Guaviare - Sentinel-2)
    process_domain(
        domain_name="target",
        image_filename=config.target_image,
        gt_filename=config.target_gt,
        config=config,
    )

    print(f"Publishing outputs {config.local_hdf5_dir} -> {config.drive_hdf5_dir}...")
    publish_outputs(config)

    print(f"\n{'=' * 60}")
    print("ALL DATASETS GENERATED SUCCESSFULLY IN HDF5 FORMAT")
    print(f"Local Output: {config.local_hdf5_dir}")
    print(f"Drive Output: {config.drive_hdf5_dir}")
    print(f"{'=' * 60}")


if __name__ == "__main__":
    main_build_base()


In [ ]:
import h5py, numpy as np, os

base = "/tmp/dares_work/hdf5_processed"  # local outputs of main_build_base()

# (subfolder, file) -> only the 6 base splits (LIME variants share masks)
files = [
    ("Source",          "source_train.h5"),
    ("Source",          "source_val.h5"),
    ("Source",          "source_test.h5"),
    ("Target_Original", "target_train.h5"),
    ("Target_Original", "target_val.h5"),
    ("Target_Original", "target_test.h5"),
]

for subdir, name in files:
    path = os.path.join(base, subdir, name)
    if not os.path.exists(path):
        print(f"[MISSING] {subdir}/{name}")
        continue

    with h5py.File(path, "r") as h:
        masks = h["masks"]
        n = masks.shape[0]
        total = 0
        hist = {}
        for i in range(n):                      # per patch (avoids loading everything into RAM)
            m = masks[i]
            vals, counts = np.unique(m, return_counts=True)
            for v, c in zip(vals.tolist(), counts.tolist()):
                hist[v] = hist.get(v, 0) + c
            total += m.size

    print(f"\n=== {subdir}/{name} | patches: {n} | pixels: {total:,} ===")
    for v in sorted(hist):
        print(f"   value {v:>3}: {hist[v]:>12,} ({100*hist[v]/total:6.2f}%)")
    if 255 not in hist:
        print("   [WARNING] value 255 MISSING: water/wetland -> 255 change NOT applied in this split")
    else:
        print(f"   [OK] 255 present: {100*hist[255]/total:.2f}% (expected ~1-2%)")

In [ ]:
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Source configuration
SOURCE_H5 = "/tmp/dares_work/hdf5_processed/Source/source_val.h5"

from pathlib import Path as _P
if not _P(SOURCE_H5).exists():
    raise FileNotFoundError(
        f"Missing {SOURCE_H5}. Run main_build_base() top-down first."
    )
IDX = 0

with h5py.File(SOURCE_H5, "r") as f:
    img = f["images"][IDX]
    mask = f["masks"][IDX]

img_hwc = np.transpose(img, (1, 2, 0))

def stretch(b):
    v_min, v_max = np.percentile(b, (2, 98))
    return np.clip((b - v_min) / (v_max - v_min), 0, 1) if v_max > v_min else np.zeros_like(b)

rgb = np.stack([stretch(img_hwc[:,:,2]), stretch(img_hwc[:,:,1]), stretch(img_hwc[:,:,0])], axis=-1)
nir = np.stack([stretch(img_hwc[:,:,3]), stretch(img_hwc[:,:,2]), stretch(img_hwc[:,:,1])], axis=-1)
cmap = ListedColormap(["#d95f02", "#1b9e77"])

# Layout synchronization: 1x3 grid without colorbar to match target grid height consistency
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)

# Standardized titles matching target cell pattern
axes[0].imshow(rgb)
axes[0].set_title("Source RGB", fontsize=12, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(nir)
axes[1].set_title("Source NIR", fontsize=12, fontweight="bold")
axes[1].axis("off")

axes[2].imshow(mask, cmap=cmap, vmin=0, vmax=1)
axes[2].set_title("Source GT", fontsize=12, fontweight="bold")
axes[2].axis("off")

# Save figure
plt.savefig("source_visualization.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# =========================================================================
# 1. IMPORTS AND CONFIGURATION
# =========================================================================

import zlib
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Tuple

import cv2
import h5py
import numpy as np
from tqdm import tqdm


# Canonical tiers: destination folder + filename suffix + severity pool.
# (The historical suffix of the medium tier is "med": Target_Medium/target_*_lime_med.h5)
LIME_TIERS: Dict[str, Dict[str, object]] = {
    "low": {"subdir": "Target_Low", "suffix": "low", "pool": (0.1, 0.2)},
    "medium": {"subdir": "Target_Medium", "suffix": "med", "pool": (0.3, 0.4, 0.5)},
    "high": {"subdir": "Target_High", "suffix": "high", "pool": (0.6, 0.7)},
}


@dataclass(frozen=True)
class LimePerturbationConfig:
    # Single source of truth, same as DatasetConfig
    drive_root: Path = DRIVE_ROOT if "DRIVE_ROOT" in dir() else Path("/content/drive/MyDrive/GEE_DARES_Dataset")
    local_root: Path = LOCAL_ROOT if "LOCAL_ROOT" in dir() else Path("/tmp/dares_work")

    # Discrete severity levels and sampling pools per level (keys = canonical tiers)
    severity_levels: Dict[str, Tuple[float, ...]] = field(
        default_factory=lambda: {
            tier: tuple(info["pool"]) for tier, info in LIME_TIERS.items()
        }
    )

    # Guided Filter hyperparameters for the illumination map T(x)
    gf_radius: int = 15
    gf_eps: float = 1e-3
    eps_div: float = 1e-6
    gamma_scale: float = 3.5

    # Seed for reproducible severity sampling
    seed: int = 42

    @property
    def local_hdf5_dir(self) -> Path:
        return self.local_root / "hdf5_processed"

    @property
    def drive_hdf5_dir(self) -> Path:
        return self.drive_root / "hdf5_processed"


# =========================================================================
# NATIVE GUIDED FILTER (NO XIMGPROC / EXTRA DEPENDENCIES)
# =========================================================================


def guided_filter_2d(
    guide: np.ndarray, src: np.ndarray, radius: int, eps: float
) -> np.ndarray:
    """Canonical Guided Filter implementation (He et al., 2013) using cv2.boxFilter.

    Args:
        guide: 2D guide image (H, W), float32.
        src: 2D input image to filter (H, W), float32.
        radius: Filter window radius.
        eps: Regularization parameter.

    Returns:
        Smoothed output array (H, W), float32.
    """
    ksize = (2 * radius + 1, 2 * radius + 1)

    # Local means
    mean_i = cv2.boxFilter(guide, cv2.CV_32F, ksize)
    mean_p = cv2.boxFilter(src, cv2.CV_32F, ksize)
    mean_ip = cv2.boxFilter(guide * src, cv2.CV_32F, ksize)
    cov_ip = mean_ip - (mean_i * mean_p)

    # Local variance of the guide
    mean_ii = cv2.boxFilter(guide * guide, cv2.CV_32F, ksize)
    var_i = mean_ii - (mean_i * mean_i)

    # Linear coefficients a and b
    a = cov_ip / (var_i + eps)
    b = mean_p - (a * mean_i)

    # Coefficient means
    mean_a = cv2.boxFilter(a, cv2.CV_32F, ksize)
    mean_b = cv2.boxFilter(b, cv2.CV_32F, ksize)

    # Guided reconstruction
    return (mean_a * guide) + mean_b


# =========================================================================
# 3. LIME RETINEX OPERATOR (MULTISPECTRAL S2)
# =========================================================================


def apply_lime_patch(
    image: np.ndarray, severity: float, config: LimePerturbationConfig
) -> np.ndarray:
    """Applies LIME illumination attenuation to a 4-band multispectral patch.

    Args:
        image: Input array (4, H, W) in [0.0, 1.0], float32.
        severity: Scalar perturbation severity factor.
        config: Operator hyperparameters instance.

    Returns:
        Array of shape (4, H, W), float32, clipped to [0.0, 1.0].
    """
    # 1. Initial illumination map T_0 (max across bands)
    t_init = np.max(image, axis=0).astype(np.float32)

    # 2. Filter guide (mean of visible bands B2, B3, B4)
    guide = np.mean(image[:3, :, :], axis=0).astype(np.float32)

    # 3. Spatial refinement with the native Guided Filter
    t_refined = guided_filter_2d(
        guide=guide,
        src=t_init,
        radius=config.gf_radius,
        eps=config.gf_eps,
    )

    # Clamp to physical transmittance bounds
    t_refined = np.clip(t_refined, 0.01, 1.0)

    # 4. Gamma-exponent modulation and attenuation factor
    gamma = 1.0 + (severity * config.gamma_scale)
    attenuation_map = np.power(t_refined, gamma) / (
        t_refined + config.eps_div
    )

    # 5. Spectral modulation and clipping to [0.0, 1.0]
    perturbed_image = image * attenuation_map[np.newaxis, :, :]
    return np.clip(perturbed_image, 0.0, 1.0).astype(np.float32)


# =========================================================================
# 4. HDF5 BATCH GENERATION PIPELINE
# =========================================================================


def ensure_base_available(config: LimePerturbationConfig) -> None:
    """Ensures Target_Original/*.h5 exist locally; downloads them from Drive if missing."""
    want = ["target_train.h5", "target_val.h5", "target_test.h5"]
    missing = [w for w in want
               if not (config.local_hdf5_dir / "Target_Original" / w).exists()]
    if not missing:
        return
    print(f"  [lime-stage] missing locally: {missing}, buscando en Drive...")
    for w in missing:
        src = config.drive_hdf5_dir / "Target_Original" / w
        dst = config.local_hdf5_dir / "Target_Original" / w
        if not src.exists():
            raise FileNotFoundError(
                f"Missing {dst}, not on Drive either ({src}). "
                "Run main_build_base() first."
            )
        dst.parent.mkdir(parents=True, exist_ok=True)
        import shutil as _sh
        _sh.copyfile(src, dst)
        print(f"  [lime-stage] downloaded {w} from Drive")


def process_hdf5_split(
    split_name: str,
    level_name: str,
    severity_pool: Tuple[float, ...],
    config: LimePerturbationConfig,
    rng: np.random.Generator,
) -> None:
    """Reads the clean Target split, perturbs its patches, and writes the degraded HDF5."""
    try:
        tier = LIME_TIERS[level_name]
    except KeyError:
        raise ValueError(f"level_name must be one of {sorted(LIME_TIERS)}, got {level_name!r}")
    subdir, suffix = str(tier["subdir"]), str(tier["suffix"])
    input_file = config.local_hdf5_dir / "Target_Original" / f"target_{split_name}.h5"
    output_file = config.local_hdf5_dir / subdir / f"target_{split_name}_lime_{suffix}.h5"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    if not input_file.exists():
        print(f"[-] Skipping {input_file.name}: file not found.")
        return

    with h5py.File(input_file, "r") as src_h5:
        images = src_h5["images"][:]
        masks = src_h5["masks"][:]

    n_samples, channels, height, width = images.shape
    perturbed_images = np.empty_like(images, dtype=np.float32)

    # Deterministic discrete severity sampling per sample
    severities = rng.choice(severity_pool, size=n_samples)

    for i in tqdm(
        range(n_samples),
        desc=f"[{split_name.upper()}] LIME-{level_name.upper()}",
        ncols=80,
    ):
        perturbed_images[i] = apply_lime_patch(
            image=images[i],
            severity=float(severities[i]),
            config=config,
        )

    # Save to HDF5 with LZF compression
    with h5py.File(output_file, "w") as dst_h5:
        dst_h5.create_dataset(
            "images",
            data=perturbed_images,
            dtype="float32",
            chunks=(1, channels, height, width),
            compression="lzf",
        )
        dst_h5.create_dataset(
            "masks",
            data=masks,
            dtype="uint8",
            chunks=(1, height, width),
            compression="lzf",
        )

    print(
        f"  -> Saved: {output_file.name} | "
        f"Samples: {n_samples} | Pool: {severity_pool}"
    )


# =========================================================================
# 5. ENTRY POINT
# =========================================================================


def main_build_lime() -> None:
    config = LimePerturbationConfig()
    splits = ["train", "val", "test"]

    print("=" * 65)
    print("LIME PERTURBATION GENERATOR FOR THE TARGET DOMAIN")
    print(f"Local: {config.local_hdf5_dir}")
    print("=" * 65)

    ensure_base_available(config)

    for level_name, severity_pool in config.severity_levels.items():
        print(
            f"\n>>> Level: {level_name.upper()} (Options: {severity_pool}) <<<"
        )
        rng = np.random.default_rng(
            config.seed + zlib.crc32(level_name.encode()) % 1000
        )

        for split in splits:
            process_hdf5_split(
                split_name=split,
                level_name=level_name,
                severity_pool=severity_pool,
                config=config,
                rng=rng,
            )

    print("\n" + "=" * 65)
    print("ALL LIME VARIANTS GENERATED SUCCESSFULLY")
    print("=" * 65)


if __name__ == "__main__":
    main_build_lime()

In [ ]:
paths = {
    "Original": "/tmp/dares_work/hdf5_processed/Target_Original/target_val.h5",
    "Low": "/tmp/dares_work/hdf5_processed/Target_Low/target_val_lime_low.h5",
    "Medium": "/tmp/dares_work/hdf5_processed/Target_Medium/target_val_lime_med.h5",
    "High": "/tmp/dares_work/hdf5_processed/Target_High/target_val_lime_high.h5"
}

IDX = 2

# Guard: the grid needs all 4 splits (LIME ones are built by main_build_lime).
_missing = [f"{k}: {v}" for k, v in paths.items() if not Path(v).exists()]
if _missing:
    raise FileNotFoundError(
        "Missing .h5 files for plotting:\n  " + "\n  ".join(_missing)
        + "\nRun the notebook top-down: main_build_base() -> main_build_lime() primero."
    )

# Reuse stretch/cmap from the Source cell if already defined (execution order)
try:
    stretch, cmap
except NameError:
    from matplotlib.colors import ListedColormap
    def stretch(b):
        import numpy as _np
        v_min, v_max = _np.percentile(b, (2, 98))
        return _np.clip((b - v_min) / (v_max - v_min), 0, 1) if v_max > v_min else _np.zeros_like(b)
    cmap = ListedColormap(["#d95f02", "#1b9e77"])

# 2 rows, 6 columns grid
fig, axes = plt.subplots(2, 6, figsize=(20, 8), constrained_layout=True)

levels = [("Original", 0, 0), ("Low", 0, 3), ("Medium", 1, 0), ("High", 1, 3)]

for name, row, col_start in levels:
    with h5py.File(paths[name], "r") as f:
        img = f["images"][IDX]
        mask = f["masks"][IDX]

    img_hwc = np.transpose(img, (1, 2, 0))
    rgb = np.stack([stretch(img_hwc[:,:,2]), stretch(img_hwc[:,:,1]), stretch(img_hwc[:,:,0])], axis=-1)
    nir = np.stack([stretch(img_hwc[:,:,3]), stretch(img_hwc[:,:,2]), stretch(img_hwc[:,:,1])], axis=-1)

    # Plot RGB
    axes[row, col_start].imshow(rgb)
    axes[row, col_start].set_title(f"{name} RGB", fontsize=10, fontweight='bold')
    axes[row, col_start].axis("off")

    # Plot NIR
    axes[row, col_start + 1].imshow(nir)
    axes[row, col_start + 1].set_title(f"{name} NIR", fontsize=10, fontweight='bold')
    axes[row, col_start + 1].axis("off")

    # Plot GT
    im_gt = axes[row, col_start + 2].imshow(mask, cmap=cmap, vmin=0, vmax=1)
    axes[row, col_start + 2].set_title(f"{name} GT", fontsize=10, fontweight='bold')
    axes[row, col_start + 2].axis("off")

# Save with high quality
plt.savefig("target_perturbation_grid.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
!ls -R /tmp/dares_work/hdf5_processed

In [ ]:
# =========================================================================
# HDF5 DATASET ZIP PACKAGING (DARES BENCHMARK)
# =========================================================================

from pathlib import Path
import shutil
import zipfile
from tqdm import tqdm

# -------------------------------------------------------------------------
# Input and output paths
# -------------------------------------------------------------------------
source_dir = Path("/tmp/dares_work/hdf5_processed")  # canonical local tree
local_zip = Path("/tmp/dares_work/hdf5_processed.zip")
output_zip = Path(
    "/content/drive/MyDrive/GEE_DARES_Dataset/hdf5_processed.zip"
)

# -------------------------------------------------------------------------
# Collecting the .h5 files
# -------------------------------------------------------------------------
if not source_dir.exists():
    raise FileNotFoundError(f"Directory not found: {source_dir}")

# Collect every file inside the subfolders
files_to_zip = sorted([f for f in source_dir.rglob("*") if f.is_file()])

print("=" * 65)
print(f"Source folder  : {source_dir}")
print(f"Destination file: {output_zip}")
print(f"Total files    : {len(files_to_zip)} .h5 files found")
print("=" * 65)

# -------------------------------------------------------------------------
# Creating the ZIP file
# -------------------------------------------------------------------------
# ZIP_STORED (no extra compression): HDF5 already uses internal LZF.
# To force standard deflate compression, use: compression=zipfile.ZIP_DEFLATED
output_zip.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(
    local_zip, "w", compression=zipfile.ZIP_STORED
) as zipf:
    for file_path in tqdm(
        files_to_zip, desc="Packing HDF5 into .zip", ncols=80
    ):
        # arcname preserves the relative path (e.g. Target_Low/target_train_lime_low.h5)
        relative_path = file_path.relative_to(source_dir)
        zipf.write(file_path, arcname=str(relative_path))

# -------------------------------------------------------------------------
# Summary and verification
# -------------------------------------------------------------------------
shutil.copyfile(local_zip, output_zip)
zip_size_gb = output_zip.stat().st_size / (1024**3)
print("\n" + "=" * 65)
print(f"PACKAGING COMPLETED SUCCESSFULLY")
print(f"Output file : {output_zip.name}")
print(f"Total size       : {zip_size_gb:.2f} GB")
print("=" * 65)

In [ ]:
import kagglehub
kagglehub.login()

In [ ]:
kagglehub.dataset_upload(
    'lucasiturriago/dares-amazon-uda',
    '/content/drive/MyDrive/GEE_DARES_Dataset/hdf5_processed.zip')